# Ad-hoc Clustering Experiments

In [ ]:
import sys
import os
from pathlib import Path

_here = Path(os.getcwd())
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src.preprocessing import preprocess_data, NUMERICAL_FEATURES
from src.clustering import fit_predict
from src.evaluation import evaluate_clustering
from src.evaluation import evaluate_clustering
from src.utils import load_subsample_indices
from IPython.display import display

_data = _root / "data"
DATA_PATH = str(_data / "hotel_bookings_course_release_v1.csv")
SUBSAMPLE_PATH = str(_data / "subsample_indices_v1_n30000_seed12345.txt")  # set to None (Python None, not string "None") for full dataset

## 1. Single Config

Edit this cell to change the experiments within this section

In [ ]:
ALGORITHM = "ikmeans"   # "kmeans" | "ikmeans" | "gmm"
K = 4
SEED = 0               # ignored for ikmeans
FEATURE_SET = "no_value_block"   # "full" | "no_value_block" | "no_context"
SCALER = "robust"    # "standard" | "robust"
N_PCA_COMPONENTS = 2

### Load and Preprocess

In [ ]:
df_full = pd.read_csv(DATA_PATH)
if SUBSAMPLE_PATH is not None:
    indices = load_subsample_indices(SUBSAMPLE_PATH)
    df = df_full.iloc[indices].reset_index(drop=True)
else:
    df = df_full.copy()

X, feature_names = preprocess_data(df, feature_set=FEATURE_SET, scaler=SCALER)

print(f"DataFrame shape : {df.shape}")
print(f"X shape         : {X.shape}")
print(f"Feature count   : {len(feature_names)}")
print(f"Features        : {feature_names}")

### Fit

In [ ]:
labels, runtime, n_iter = fit_predict(X, K, SEED, ALGORITHM)

sizes = pd.Series(labels, name="size").value_counts().sort_index()
sizes.index.name = "cluster"
print(f"Algorithm : {ALGORITHM}  k={K}  seed={SEED}")
print(sizes.to_string())

### Evaluate Internal Indices

In [ ]:
scores = evaluate_clustering(X, labels)
print(f"Silhouette (↑)         : {scores['silhouette']:.4f}")
print(f"Davies-Bouldin (↓)     : {scores['davies_bouldin']:.4f}")
print(f"Calinski-Harabasz (↑)  : {scores['calinski_harabasz']:.4f}")

### Cluster Profiles

Mean and std of raw (pre-scaling) numerical features per cluster.

In [ ]:
num_cols_present = [c for c in NUMERICAL_FEATURES if c in df.columns]
profile_df = df.drop_duplicates()[num_cols_present]
profile_df["cluster"] = labels

mean_profile = profile_df.groupby("cluster")[num_cols_present].mean().round(3)
std_profile  = profile_df.groupby("cluster")[num_cols_present].std().round(3)

print("=== Mean per cluster ===")
display(mean_profile)
print("\n=== Std per cluster ===")
display(std_profile)

### 2D PCA Scatter

In [ ]:
assert N_PCA_COMPONENTS >= 2, "N_PCA_COMPONENTS must be >= 2 for 2D scatter"
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=labels, cmap="tab10", alpha=0.4, s=5
)
plt.colorbar(scatter, ax=ax, label="Cluster")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title(f"{ALGORITHM}  k={K}  {FEATURE_SET}  {SCALER}  seed={SEED}")
plt.tight_layout()
plt.show()

## 2. Multi-Config Comparison

In [ ]:
#Define any set of configs in `CONFIGS`, run the cell below.

df_base = pd.read_csv(DATA_PATH)
if SUBSAMPLE_PATH is not None:
    _idx = load_subsample_indices(SUBSAMPLE_PATH)
    df_base = df_base.iloc[_idx].reset_index(drop=True)
print(f"Base dataset: {df_base.shape}")


# --- Feature-set comparison ---
_fs_configs = [
    {"algorithm": "kmeans", "k": 4, "seed": 0, "feature_set": "full",            "scaler": "standard"},
    {"algorithm": "kmeans", "k": 4, "seed": 0, "feature_set": "no_value_block",  "scaler": "standard"},
    {"algorithm": "kmeans", "k": 4, "seed": 0, "feature_set": "no_context",      "scaler": "standard"},
    {"algorithm": "kmeans", "k": 4, "seed": 0, "feature_set": "complexity_only", "scaler": "standard"},
]

# --- k comparison ---
_k_configs = [
    {"algorithm": "kmeans", "k": 3, "seed": 0, "feature_set": "full", "scaler": "standard"},
    {"algorithm": "kmeans", "k": 4, "seed": 0, "feature_set": "full", "scaler": "standard"},
    {"algorithm": "kmeans", "k": 5, "seed": 0, "feature_set": "full", "scaler": "standard"},
    {"algorithm": "kmeans", "k": 6, "seed": 0, "feature_set": "full", "scaler": "standard"},
]

# --- Scaler comparison ---
_scaler_configs = [
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "standard"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_context", "scaler": "standard"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "complexity_only", "scaler": "standard"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_context", "scaler": "robust"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "complexity_only", "scaler": "robust"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "minmax"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_context", "scaler": "minmax"},
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "complexity_only", "scaler": "minmax"},
    #{"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "mean"},
    #{"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_context", "scaler": "mean"},
    #{"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "complexity_only", "scaler": "mean"}, # winsorization + log transform seem result in errors with mean-range scaling

]

_algorithm_configs = [
    {"algorithm": "kmeans", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"},
    {"algorithm": "kmeans", "k": 3, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"},
    {"algorithm": "ikmeans", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"},
    {"algorithm": "ikmeans", "k": 3, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"},
    {"algorithm": "gmm", "k": 2, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"},
    {"algorithm": "gmm", "k": 3, "seed": 0, "feature_set": "no_value_block", "scaler": "robust"}
]


CONFIGS = _scaler_configs

In [ ]:
_results = []
for cfg in CONFIGS:
    X_c, _= preprocess_data(df_base, feature_set=cfg["feature_set"], scaler=cfg["scaler"])
    labels_c, _, _ = fit_predict(X_c, cfg["k"], cfg["seed"], cfg["algorithm"])
    scores_c = evaluate_clustering(X_c, labels_c)
    label = f"{cfg['algorithm']} k={cfg['k']} | {cfg['feature_set']} | {cfg['scaler']}"
    _results.append({"label": label, "cfg": cfg, "X": X_c, "labels": labels_c, **scores_c})
    print(f"[done] {label}")

In [ ]:
scores_df = pd.DataFrame([
    {
        "config": r["label"],
        "silhouette ↑": round(r["silhouette"], 4),
        "davies_bouldin ↓": round(r["davies_bouldin"], 4),
        "calinski_harabasz ↑": round(r["calinski_harabasz"], 4),
    }
    for r in _results
]).set_index("config")

display(scores_df)

In [ ]:
_n = len(_results)
_ncols = min(_n, 3)
_nrows = (_n + _ncols - 1) // _ncols

fig2d, axes2d = plt.subplots(_nrows, _ncols, figsize=(6 * _ncols, 5 * _nrows), squeeze=False)
axes_flat = axes2d.flatten()

for i, r in enumerate(_results):
    pca2 = PCA(n_components=2, random_state=0)
    X_2d = pca2.fit_transform(r["X"])
    ax = axes_flat[i]
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=r["labels"], cmap="tab10", alpha=0.4, s=3)
    plt.colorbar(sc, ax=ax, label="Cluster")
    ax.set_title(r["label"], fontsize=9)
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("2D PCA, Multi-Config Comparison", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
_n = len(_results)
_ncols = min(_n, 3)
_nrows = (_n + _ncols - 1) // _ncols

fig3d = plt.figure(figsize=(7 * _ncols, 6 * _nrows))
for i, r in enumerate(_results):
    pca3 = PCA(n_components=3, random_state=0)
    X_3d = pca3.fit_transform(r["X"])
    ax3 = fig3d.add_subplot(_nrows, _ncols, i + 1, projection="3d")
    ax3.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2],
                c=r["labels"], cmap="tab10", alpha=0.3, s=3)
    ax3.set_title(r["label"], fontsize=9)
    ax3.set_xlabel(f"PC1 ({pca3.explained_variance_ratio_[0]:.1%})")
    ax3.set_ylabel(f"PC2 ({pca3.explained_variance_ratio_[1]:.1%})")
    ax3.set_zlabel(f"PC3 ({pca3.explained_variance_ratio_[2]:.1%})")

plt.suptitle("3D PCA, Multi-Config Comparison", y=1.01)
plt.tight_layout()
plt.show()

## Elbow Curve

In [ ]:
def _sse(X, labels):
    """Manual SSE, equivalent to KMeans inertia. Used because fit_predict returns (labels, runtime, n_iter), not the model."""
    sse = 0.0
    for k in np.unique(labels):
        pts = X[labels == k]
        sse += float(((pts - pts.mean(axis=0)) ** 2).sum())
    return sse

In [ ]:
_elbow_ks = list(range(2, 9))
_elbow_sse = []
for _k in _elbow_ks:
    _lbl, _, _ = fit_predict(X, _k, SEED, ALGORITHM)
    _elbow_sse.append(_sse(X, _lbl))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(_elbow_ks, _elbow_sse, marker="o", linestyle="-")
ax.set_title(f"Elbow curve, {ALGORITHM} | {FEATURE_SET} | {SCALER}")
ax.set_xlabel("K")
ax.set_xticks(_elbow_ks)
ax.set_ylabel("SSE (inertia)")
plt.tight_layout()
plt.show()


In [ ]:
_unique_fs_sc = sorted({(c["feature_set"], c["scaler"]) for c in _scaler_configs})

_ncols = 3
_nrows = -(-len(_unique_fs_sc) // _ncols)
fig, axes = plt.subplots(_nrows, _ncols, figsize=(6 * _ncols, 4 * _nrows), squeeze=False)
_axes_flat = axes.flatten()

for i, (fs, sc) in enumerate(_unique_fs_sc):
    _X_e, _ = preprocess_data(df_base, feature_set=fs, scaler=sc)
    _sse_vals = []
    for _k in _elbow_ks:
        _lbl, _, _ = fit_predict(_X_e, _k, 0, "kmeans")
        _sse_vals.append(_sse(_X_e, _lbl))
    _axes_flat[i].plot(_elbow_ks, _sse_vals, marker="o", linestyle="-")
    _axes_flat[i].set_title(f"{fs} | {sc}", fontsize=9)
    _axes_flat[i].set_xlabel("K")
    _axes_flat[i].set_xticks(_elbow_ks)
    _axes_flat[i].set_ylabel("SSE (inertia)")

for j in range(i + 1, len(_axes_flat)):
    _axes_flat[j].set_visible(False)

plt.suptitle("Elbow curves, K-Means | scaler configs", y=1.02)
plt.tight_layout()
plt.show()


## KMeans vs iKmeans

In [ ]:
_SWEEP_FEATURE_SET = "no_value_block"
_SWEEP_SCALER = "robust"
_K_VALUES = list(range(2, 9))

_sweep_configs = (
    [{"algorithm": "kmeans",  "k": k, "seed": 0, "feature_set": _SWEEP_FEATURE_SET, "scaler": _SWEEP_SCALER} for k in _K_VALUES] +
    [{"algorithm": "ikmeans", "k": k, "seed": 0, "feature_set": _SWEEP_FEATURE_SET, "scaler": _SWEEP_SCALER} for k in _K_VALUES]
)

_sweep_results = []
for cfg in _sweep_configs:
    X_c, _ = preprocess_data(df_base, feature_set=cfg["feature_set"], scaler=cfg["scaler"])
    _lbl, _rt, _ = fit_predict(X_c, cfg["k"], cfg["seed"], cfg["algorithm"])
    _scores = evaluate_clustering(X_c, _lbl)
    _sweep_results.append({
        "algorithm": cfg["algorithm"], "K": cfg["k"],
        "sse": _sse(X_c, _lbl), "runtime_s": _rt,
        "X": X_c, "labels": _lbl, **_scores,
    })
    print(f"[done] {cfg['algorithm']} k={cfg['k']}")

_sweep_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("X", "labels")} for r in _sweep_results])
display(_sweep_df.round(4))


In [ ]:
_metrics_cfg = [
    ("silhouette", "Silhouette", "higher is better"),
    ("calinski_harabasz", "Calinski-Harabasz", "higher is better"),
    ("davies_bouldin", "Davies-Bouldin", "lower is better"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, label, better) in zip(axes, _metrics_cfg):
    for algo, ls, mk in [("kmeans", "-", "o"), ("ikmeans", "--", "s")]:
        sub = _sweep_df[_sweep_df["algorithm"] == algo]
        ax.plot(sub["K"], sub[col], marker=mk, linestyle=ls, label=algo)
    ax.set_title(f"{label}\n({better})")
    ax.set_xlabel("K")
    ax.set_xticks(_K_VALUES)
    ax.set_ylabel("score")
    ax.legend(frameon=True)

plt.suptitle(f"Metric curves, kmeans vs ikmeans | {_SWEEP_FEATURE_SET} | {_SWEEP_SCALER}", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for algo, ls, mk in [("kmeans", "-", "o"), ("ikmeans", "--", "s")]:
    sub = _sweep_df[_sweep_df["algorithm"] == algo]
    ax.plot(sub["K"], sub["runtime_s"], marker=mk, linestyle=ls, label=algo)
ax.set_title("Runtime per K, K-Means vs IK-Means")
ax.set_xlabel("K")
ax.set_xticks(_K_VALUES)
ax.set_ylabel("wall-clock time (s)")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()
